In [68]:
import pandas as pd

data=pd.read_json("News_Category_Dataset_v3.json",lines=True)
data.shape

(209527, 6)

In [69]:
data.head()

,link,headline,category,short_description,authors,date
0,https://www.huffpost.com/entry/covid-boosters-...,Over 4 Million Americans Roll Up Sleeves For O...,U.S. NEWS,Health experts said it is too early to predict...,"Carla K. Johnson, AP",2022-09-23
1,https://www.huffpost.com/entry/american-airlin...,"American Airlines Flyer Charged, Banned For Li...",U.S. NEWS,He was subdued by passengers and crew when he ...,Mary Papenfuss,2022-09-23
2,https://www.huffpost.com/entry/funniest-tweets...,23 Of The Funniest Tweets About Cats And Dogs ...,COMEDY,"""Until you have a dog you don't understand wha...",Elyse Wanshel,2022-09-23
3,https://www.huffpost.com/entry/funniest-parent...,The Funniest Tweets From Parents This Week (Se...,PARENTING,"""Accidentally put grown-up toothpaste on my to...",Caroline Bologna,2022-09-23
4,https://www.huffpost.com/entry/amy-cooper-lose...,Woman Who Called Cops On Black Bird-Watcher Lo...,U.S. NEWS,Amy Cooper accused investment firm Franklin Te...,Nina Golgowski,2022-09-22


In [70]:
data=data['short_description']
data=data.head(20000)

In [71]:
data[0]

'Health experts said it is too early to predict whether demand would match up with the 171 million doses of the new boosters the U.S. ordered for the fall.'

In [72]:
data.duplicated().sum()
data.drop_duplicates(inplace=True)

In [73]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [74]:
tokenizer=Tokenizer(oov_token='<OOV>')
tokenizer.fit_on_texts(data)
data=tokenizer.texts_to_sequences(data)

In [75]:
input_sequences=[]
for seq in data:
    if len(seq) < 2:
        continue
    for i in range(1,len(seq)):
        ngram=seq[:i+1]
        input_sequences.append(ngram)

In [76]:
max_len=max([len(i) for i in input_sequences])
input=pad_sequences(input_sequences,padding='pre',maxlen=max_len)

In [77]:
X=input[:,:-1]
y=input[:,-1]
y=np.array(y)

In [78]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
vocab_size=len(tokenizer.word_index)+1
max_len=X.shape[1]

In [79]:
model=Sequential([
    Embedding(vocab_size,128,input_length=max_len),
    LSTM(256),
    Dense(vocab_size,activation='softmax')
])

C:\Users\hp\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [80]:
model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])
model.build(input_shape=(None,max_len))
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 43, 128)        │     3,372,032 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 256)            │       394,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 26344)          │     6,770,408 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,536,680 (40.19 MB)

 Trainable params: 10,536,680 (40.19 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.fit(X_train,y_train,validation_data=(X_test,y_test),epochs=10,batch_size=128)

Epoch 1/10
  95/1876 ━━━━━━━━━━━━━━━━━━━━ 6:25 216ms/step - accuracy: 0.0587 - loss: 7.5920

InvalidArgumentError: Graph execution error:

Detected at node compile_loss/sparse_categorical_crossentropy/SparseSoftmaxCrossEntropyWithLogits/SparseSoftmaxCrossEntropyWithLogits defined at (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main

  File "<frozen runpy>", line 88, in _run_code

  File "C:\Users\hp\AppData\Roaming\Python\Python313\site-packages\ipykernel_launcher.py", line 18, in <module>

  File "C:\Users\hp\AppData\Roaming\Python\Python313\site-packages\traitlets\config\application.py", line 1082, in launch_instance

  File "C:\Users\hp\AppData\Roaming\Python\Python313\site-packages\ipykernel\kernelapp.py", line 807, in start

  File "C:\Users\hp\AppData\Roaming\Python\Python313\site-packages\tornado\platform\asyncio.py", line 211, in start

  File "c:\Users\hp\anaconda3\Lib\asyncio\base_events.py", line 683, in run_forever

  File "c:\Users\hp\anaconda3\Lib\asyncio\base_events.py", line 2050, in _run_once

  File "c:\Users\hp\anaconda3\Lib\